# Notebook 02 - Memory Lab

Demonstrates all three memory tiers.

- **Tier 1** ConversationMemory - rolling window
- **Tier 2** CatalogVectorStore - Qdrant semantic search with allergen exclusion
- **Tier 3** RecipientMemory - profile lookup + formatting

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

from config.settings import Settings
settings = Settings()
print('Settings loaded ✅')

Settings loaded ✅


In [2]:
# ── Tier 1: ConversationMemory ──────────────────────────
from src.memory.short_term import ConversationMemory

mem = ConversationMemory(max_messages=10)
mem.add_message('user', 'I want to buy a gift for my wife Amaya for her birthday')
mem.add_message('assistant', 'Happy to help! What is Amaya interested in?')
mem.add_message('user', 'She loves dark chocolate but is allergic to nuts')

print('=== Last 3 messages ===')
for m in mem.get_last_n(3):
    print(f"  [{m['role']:9s}] {m['content']}")

print(f'\n=== Context summary ===' )
print(mem.get_context_summary())

d:\Zuu Crew Agentic AI\Projects\Mini Project 03\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Last 3 messages ===
  [user     ] I want to buy a gift for my wife Amaya for her birthday
  [assistant] Happy to help! What is Amaya interested in?
  [user     ] She loves dark chocolate but is allergic to nuts

=== Context summary ===
USER: I want to buy a gift for my wife Amaya for her birthday
ASSISTANT: Happy to help! What is Amaya interested in?
USER: She loves dark chocolate but is allergic to nuts


In [3]:
# ── Tier 3: RecipientMemory ─────────────────────────────
from src.memory.semantic import RecipientMemory

semantic = RecipientMemory(settings.PROFILES_PATH)
print('Known recipients:', semantic.list_recipients())

# Fuzzy match
for query in ['amma', 'my wife', 'boss', 'thaththa', 'Amaya']:
    result = semantic.find_recipient(query)
    if result:
        key, profile = result
        print(f'  "{query}" → {key} ({profile["name"]})')
    else:
        print(f'  "{query}" → Not found')

Known recipients: ['wife', 'mother', 'boss', 'daughter', 'friend_dinesh']
  "amma" → mother (Kamala)
  "my wife" → wife (Amaya)
  "boss" → boss (Mr. Rajapakse)
  "thaththa" → Not found
  "Amaya" → wife (Amaya)


In [4]:
# ── Profile format for LLM ──────────────────────────────
print(semantic.format_for_prompt('wife'))
print('---')
print(semantic.format_for_prompt('mother'))

Recipient: Amaya (wife)
Preferences: sushi, gold jewelry, spa gift sets, dark chocolate, red roses
Dislikes: white chocolate, carnations
Allergies: peanut, peanuts, nuts, shellfish  ⚠️ CRITICAL - these MUST be filtered from all recommendations
Dietary: no-nuts, pescatarian
Location: Colombo
Budget: LKR 3,000 – 15,000
Past gifts: Nut-Free Chocolate Cake (birthday 2024), Red Rose Bouquet (24 stems) (valentines 2024)
Notes: Prefers elegant packaging. Loves surprises at her office.
---
Recipient: Kamala (mother)
Preferences: fruit baskets, traditional sweets, sarees, tea sets, religious items
Dislikes: too modern gifts, perfume
Allergies: gluten  ⚠️ CRITICAL - these MUST be filtered from all recommendations
Dietary: gluten-free, vegetarian
Location: Kandy
Budget: LKR 2,000 – 8,000
Past gifts: Premium Fruit Basket (mothers_day 2024)
Notes: Lives alone. Delivery to Kandy address. Prefers morning delivery.


In [5]:
# ── Tier 2: CatalogVectorStore (requires Qdrant) ────────
# NOTE: Run setup_ingest.py first to populate Qdrant.

from src.memory.long_term import CatalogVectorStore

store = CatalogVectorStore(settings)
try:
    info = store.get_collection_info()
    print(f'Collection: {info}')
except Exception as e:
    print(f'Qdrant not ready: {e}')
    print('Run: python setup_ingest.py --force-fallback')

📦 Loading embedding model: all-MiniLM-L6-v2…


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1590.01it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model ready
Collection: {'name': 'kapruka_catalog', 'points_count': 17305, 'status': 'green'}


In [6]:
# ── Allergen-safe search demo ───────────────────────────
# (Requires Qdrant to be populated)
try:
    print('=== Search: birthday chocolate (allergen-safe for wife - nut allergy) ===')
    results = store.search_excluding_allergens(
        query='birthday chocolate gift',
        allergens=['nuts'],
        top_k=3,
    )
    for r in results:
        p = r['product']
        print(f"  [{r['score']:.3f}] {p['product_name']} | LKR {p['price_lkr']:,.0f}")
        print(f"         Allergens: {p['contains_allergens'] or 'None'}")
except Exception as e:
    print(f'Skipping (Qdrant not ready): {e}')

=== Search: birthday chocolate (allergen-safe for wife - nut allergy) ===
  [0.533] Happy Birthday Milestone 60 Card — Luxury Handmade (English) | LKR 1,200
         Allergens: None
  [0.520] Happy Birthday Milestone 60 Card — Mini Postcard (English) | LKR 350
         Allergens: None
  [0.518] Happy Birthday Milestone 21 Card — Luxury Handmade (English) | LKR 1,200
         Allergens: None
